# 2장. Transformer 구조 이해

이 장은 PDF 교재의 `2. 트랜스포머(Transformer)` 전체와 `LLM/llm.ipynb`의 Transformer 직접 구현 실습을 통합한 장입니다.

## 이 장에서 다루는 내용

1. Transformer의 핵심 개념
2. Attention과 Self-Attention
3. RNN과 Transformer의 처리 방식 차이
4. Encoder와 Decoder 구조
5. Q, K, V의 역할
6. Scaled Dot-Product Attention 수식
7. 입력 임베딩과 위치 인코딩
8. Multi-Head Attention
9. Add & Norm, Feed Forward Network
10. Masked Multi-Head Attention
11. BERT와 GPT의 차이
12. TensorFlow/Keras로 Tiny Transformer 감정 분류 모델 구현
13. 학습 루프와 추론 실습

1장에서 문장이 토큰 ID와 임베딩 벡터로 바뀌는 과정을 다뤘습니다. 2장에서는 그 임베딩 벡터들이 Transformer 내부에서 어떻게 서로의 관계를 계산하고, 최종 예측으로 이어지는지 살펴봅니다.


## 2.1 Transformer의 핵심 아이디어

Transformer는 문장의 모든 단어 사이의 관계를 한 번에 계산하는 딥러닝 구조입니다. PDF 교재에서는 Transformer의 핵심을 다음 세 가지로 설명합니다.

| 핵심 개념 | 설명 |
|---|---|
| Attention 메커니즘 | 문장의 모든 단어 간 연관성을 계산하고 중요한 단어에 더 집중합니다. |
| Self-Attention | 한 문장 안의 단어들이 서로 얼마나 관련 있는지 계산합니다. |
| 병렬 처리 | RNN처럼 단어를 순서대로 하나씩 처리하지 않고 여러 단어를 동시에 처리할 수 있습니다. |

예를 들어 `고양이가 나무 위로 올라갔다`라는 문장에서 `고양이`와 `올라갔다`는 의미적으로 강하게 연결됩니다. Transformer는 이런 관계를 Attention으로 계산합니다.

또 `나는 사과를 좋아한다`라는 문장에서 `사과`는 `좋아한다`와 관계가 큽니다. Self-Attention은 같은 문장 안의 단어들이 서로 어떤 관계인지 파악합니다.


## 2.2 RNN과 Transformer의 차이

기존 RNN 계열 모델은 문장을 순서대로 처리합니다.

```text
나는 -> 사과를 -> 좋아한다
```

즉 앞 단어를 처리한 결과가 다음 단어 처리에 영향을 줍니다. 이 방식은 순서를 자연스럽게 반영하지만, 긴 문장을 처리할 때 정보가 희미해지고 병렬 처리가 어렵습니다.

Transformer는 문장의 모든 단어를 동시에 보고, 단어 간 관계를 Attention으로 계산합니다.

```text
나는, 사과를, 좋아한다 -> 단어 간 관계를 한 번에 계산
```

PDF 교재의 비유처럼 RNN은 레고 블록을 하나씩 쌓는 과정이고, Transformer는 여러 블록을 한꺼번에 쌓는 과정에 가깝습니다. 그래서 대규모 데이터 학습과 추론에서 Transformer가 훨씬 효율적입니다.


## 2.3 Encoder와 Decoder

Transformer는 크게 Encoder와 Decoder로 나눌 수 있습니다.

| 구성 | 역할 | PDF 교재의 비유 | 예시 작업 |
|---|---|---|---|
| Encoder | 입력 문장을 이해하고 의미 있는 숫자 벡터 표현으로 변환 | 요약 담당 | 텍스트 요약, 이미지 특징 추출, 질문 의도 파악 |
| Decoder | Encoder가 만든 표현을 바탕으로 새 문장을 생성 | 번역 담당 | 번역, 텍스트 생성, 이미지 설명 생성, 답변 생성 |

Encoder는 입력 문장을 읽고 의미를 벡터로 압축합니다. Decoder는 그 벡터 정보를 이용해 출력 문장을 한 단어씩 생성합니다.

모델 종류에 따라 Encoder만 쓰거나, Decoder만 쓰거나, 둘 다 쓰기도 합니다.

- BERT: Encoder 중심 모델
- GPT: Decoder 중심 모델
- 원래 Transformer 번역 모델: Encoder와 Decoder 모두 사용


## 2.4 Q, K, V의 역할

PDF 교재는 Q, K, V를 다음처럼 설명합니다.

| 요소 | 이름 | 의미 |
|---|---|---|
| Q | Query | 나는 지금 무엇이 궁금한가 |
| K | Key | 나는 어떤 특징이나 정보를 가지고 있는가 |
| V | Value | 실제로 전달할 정보는 무엇인가 |

문장 예시:

```text
나는 어제 사과를 먹었다
```

동작 흐름은 다음과 같습니다.

1. 각 단어를 벡터로 변환합니다.
2. 각 단어 벡터에서 Q, K, V를 만듭니다.
3. 특정 단어의 Q와 모든 단어의 K를 비교합니다.
4. 유사도, 즉 Attention Score를 계산합니다.
5. Softmax로 중요도 가중치를 만듭니다.
6. 가중치로 V들을 곱하고 합산해 Context Vector를 만듭니다.

핵심은 `현재 단어가 문장 안의 다른 단어들을 얼마나 참고해야 하는가`를 계산하는 것입니다.


## 2.5 Scaled Dot-Product Attention 수식

PDF의 Q/K/V 도식에서 핵심 수식은 다음입니다.

$$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

각 항목의 의미는 다음과 같습니다.

| 항목 | 의미 |
|---|---|
| `QK^T` | Query와 Key의 유사도를 행렬 곱으로 계산합니다. |
| `sqrt(d_k)` | Key 벡터 차원의 제곱근으로 나눠 값이 너무 커지는 것을 막습니다. |
| `softmax` | 유사도 점수를 확률 형태의 가중치로 바꿉니다. |
| `V` | 실제 전달할 정보입니다. |
| 최종 결과 | 문맥이 반영된 Context Vector입니다. |

이 장의 실습에서는 이 수식을 TensorFlow 코드로 직접 구현합니다.


## 2.6 Encoder 내부 구성

PDF 교재의 Encoder 구성은 다음과 같습니다.

### 입력 임베딩, Input Embedding

- 각 단어를 숫자 벡터로 변환합니다.
- 흐름은 토크나이징 -> 인코딩 -> 임베딩입니다.
- 벡터 간 거리는 의미적 유사성과 연결됩니다.

### 위치 인코딩, Positional Encoding

- Transformer는 기본적으로 단어 순서를 따로 알지 못합니다.
- 그래서 단어가 문장에서 몇 번째인지 알려주는 위치 정보를 추가합니다.

### Encoder Layer

- Multi-Head Attention: 단어들이 서로 어떻게 관련 있는지 계산합니다.
- Feed Forward Network: 각 단어 벡터를 더 깊게 처리합니다.
- Add & Norm: 잔차 연결과 정규화로 학습을 안정화합니다.


## 2.7 Decoder 내부 구성

PDF 교재의 Decoder 구성은 다음과 같습니다.

### 출력 임베딩과 위치 인코딩

- 지금까지 생성된 출력 단어를 벡터로 바꿉니다.
- 위치 인코딩을 추가해 출력 문장 안의 순서를 반영합니다.

### Masked Multi-Head Attention

- Decoder는 문장을 한 단어씩 생성합니다.
- 미래 단어를 미리 보면 안 되므로 마스크 처리를 합니다.
- 예를 들어 `I eat`을 생성 중이라면, `eat`을 예측할 때 이후 단어는 볼 수 없습니다.

### Encoder-Decoder Attention과 Feed Forward

- Encoder가 이해한 입력 문장의 정보와 현재 Decoder 출력 상태를 연결합니다.
- Feed Forward와 Add & Norm을 거쳐 다음 토큰 예측으로 이어집니다.


## 2.8 BERT와 GPT 차이

PDF 교재의 BERT와 GPT 비교를 정리하면 다음과 같습니다.

| 특징 | BERT | GPT |
|---|---|---|
| 방향성 | 양방향 | 단방향 |
| 학습 목표 | 다양한 자연어처리 태스크 | 다음 단어 예측 |
| 강점 | 문맥 이해, 다양한 NLP 작업 | 텍스트 생성, 챗봇, 번역 |
| 단점 | 모델이 크고 복잡하며 학습 시간이 오래 걸림 | 단방향이어서 문맥 이해가 제한될 수 있음 |

BERT는 문장의 앞뒤 문맥을 함께 봅니다. 그래서 문장 이해, 분류, 개체명 인식, 마스크 채우기 같은 작업에 강합니다.

GPT는 이전 토큰들을 보고 다음 토큰을 예측합니다. 그래서 자연스러운 문장 생성, 대화, 글쓰기, 코드 생성에 강합니다.


## 2.9 실습 개요: Tiny Transformer 감정 분류기

`LLM/llm.ipynb`에는 Transformer 구조를 이해하기 위한 작은 감정 분류 예제가 있습니다.

실습 목표:

- 6개의 짧은 영어 문장을 준비합니다.
- 긍정 문장은 1, 부정 문장은 0으로 라벨링합니다.
- 직접 단어 사전을 만듭니다.
- 문장을 토큰 ID 배열로 바꿉니다.
- Embedding과 Positional Embedding을 더합니다.
- Q, K, V를 직접 계산합니다.
- Self-Attention을 구현합니다.
- Feed Forward와 LayerNorm을 거칩니다.
- 마지막 토큰 표현으로 긍정/부정을 분류합니다.
- 새 문장에 대해 추론합니다.

이 예제는 실제 LLM에 비하면 매우 작지만, Transformer의 핵심 계산 흐름을 직접 눈으로 확인하기에 적합합니다.


In [ ]:
# TensorFlow는 모델과 신경망 레이어를 만들기 위해 사용합니다.
# layers는 Embedding, Dense, LayerNormalization 같은 Keras 레이어를 제공합니다.
# numpy는 난수 시드 고정과 수치 계산에 사용합니다.
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np


In [ ]:
# 결과를 재현하기 위해 TensorFlow와 NumPy의 난수 시드를 고정합니다.
tf.random.set_seed(42)
np.random.seed(42)


## 2.10 데이터셋과 하이퍼파라미터

원본 노트북은 긍정 문장 3개와 부정 문장 3개를 사용합니다.

| 문장 | 라벨 |
|---|---|
| `i love this movie` | 1, 긍정 |
| `this movie is great` | 1, 긍정 |
| `i liked this film` | 1, 긍정 |
| `i hate this movie` | 0, 부정 |
| `this movie is bad` | 0, 부정 |
| `i disliked this film` | 0, 부정 |

하이퍼파라미터는 다음과 같습니다.

- `seq_len = 4`: 모든 문장의 길이를 4로 맞춥니다.
- `d_model = 8`: 각 토큰을 8차원 벡터로 표현합니다.
- Attention head는 직접 구현 예제에서는 Q/K/V Dense를 하나씩 사용합니다.


In [ ]:
# 1. Dataset & Hyperparameters
# 학습용 원문 문장 6개를 준비합니다.
raw_sentences = [
    "i love this movie",
    "this movie is great",
    "i liked this film",
    "i hate this movie",
    "this movie is bad",
    "i disliked this film"
]

# 긍정 문장은 1, 부정 문장은 0으로 라벨링합니다.
labels = tf.constant([[1], [1], [1], [0], [0], [0]], dtype=tf.float32)

# 단어 사전을 직접 만듭니다. 0은 패딩 토큰으로 남겨둡니다.
vocab = {"<PAD>": 0}

# 모든 문장을 순회하면서 처음 보는 단어에 새 ID를 부여합니다.
for sentence in raw_sentences:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = len(vocab)

# vocabulary 크기, 시퀀스 길이, 임베딩 차원을 설정합니다.
vocab_size = len(vocab)
seq_len = 4
d_model = 8

# 문장을 토큰 ID 배열로 변환합니다.
input_data = []
for sentence in raw_sentences:
    ids = [vocab[word] for word in sentence.split()]
    input_data.append(ids)

# 입력 데이터를 TensorFlow 정수 텐서로 변환합니다.
input_ids = tf.constant(input_data, dtype=tf.int32)

# 구성 결과를 확인합니다.
print("단어 사전:", vocab)
print("입력 데이터 shape:", input_ids.shape)
print(input_ids)


## 2.11 Transformer 레이어 정의

이제 Transformer의 주요 구성요소를 Keras 레이어로 정의합니다.

| 코드 요소 | Transformer 개념 |
|---|---|
| `token_embedding_layer` | 입력 임베딩 |
| `pos_embedding_layer` | 위치 인코딩/위치 임베딩 |
| `W_q`, `W_k`, `W_v` | Query, Key, Value 변환 |
| `tf.matmul(q, k, transpose_b=True)` | Q와 K의 유사도 계산 |
| `tf.nn.softmax(scores)` | Attention 가중치 계산 |
| `tf.matmul(weights, v)` | Value 가중합, Context Vector |
| `layer_norm_1`, `layer_norm_2` | Add & Norm |
| `ffn_layer1`, `ffn_layer2` | Feed Forward Network |
| `classification_layer` | 긍정/부정 분류 출력 |


In [ ]:
# 2. Model Layers Definition
# 토큰 ID를 d_model 차원의 벡터로 바꾸는 임베딩 층입니다.
token_embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=d_model)

# 각 위치 0, 1, 2, 3을 나타내는 인덱스를 만듭니다.
position_indices = tf.range(start=0, limit=seq_len, delta=1)

# 위치 정보를 d_model 차원의 벡터로 바꾸는 위치 임베딩 층입니다.
pos_embedding_layer = layers.Embedding(input_dim=seq_len, output_dim=d_model)

# Self-Attention에서 사용할 Query, Key, Value 변환 레이어입니다.
W_q = layers.Dense(d_model)
W_k = layers.Dense(d_model)
W_v = layers.Dense(d_model)

# Add & Norm에 사용할 Layer Normalization입니다.
layer_norm_1 = layers.LayerNormalization(epsilon=1e-6)
layer_norm_2 = layers.LayerNormalization(epsilon=1e-6)

# Feed Forward Network입니다. 8차원 -> 16차원 -> 8차원으로 변환합니다.
ffn_layer1 = layers.Dense(units=16, activation='relu')
ffn_layer2 = layers.Dense(units=d_model)

# 마지막 토큰 표현을 기반으로 긍정/부정 확률을 출력하는 분류 층입니다.
classification_layer = layers.Dense(units=1, activation='sigmoid')


## 2.12 정방향 계산 함수

이 함수는 Transformer 블록의 핵심 흐름을 한 번에 실행합니다.

처리 순서:

1. 토큰 임베딩을 구합니다.
2. 위치 임베딩을 구합니다.
3. 둘을 더해 입력 표현 `x`를 만듭니다.
4. `x`에서 Q, K, V를 만듭니다.
5. `QK^T / sqrt(d_model)`로 Attention Score를 계산합니다.
6. Softmax로 Attention Weight를 만듭니다.
7. Weight와 V를 곱해 Attention Output을 만듭니다.
8. Residual Connection과 LayerNorm을 적용합니다.
9. Feed Forward Network를 통과합니다.
10. 다시 Residual Connection과 LayerNorm을 적용합니다.
11. 마지막 토큰 벡터를 사용해 긍정/부정 확률을 출력합니다.


In [ ]:
# Transformer의 정방향 계산을 함수로 정의합니다.
def transformer_forward(ids):
    # 토큰 ID를 임베딩 벡터로 변환합니다.
    w_emb = token_embedding_layer(ids)

    # 위치 인덱스를 위치 임베딩 벡터로 변환합니다.
    p_emb = pos_embedding_layer(position_indices)

    # 토큰 임베딩과 위치 임베딩을 더합니다.
    # p_emb는 문장 하나 기준 shape이므로 batch 차원을 맞추기 위해 expand_dims를 사용합니다.
    x = w_emb + tf.expand_dims(p_emb, axis=0)

    # 입력 x에서 Query, Key, Value를 계산합니다.
    q = W_q(x)
    k = W_k(x)
    v = W_v(x)

    # Query와 Key의 내적으로 Attention Score를 계산합니다.
    scores = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(tf.cast(d_model, tf.float32))

    # Score를 softmax에 통과시켜 Attention Weight로 바꿉니다.
    weights = tf.nn.softmax(scores, axis=-1)

    # Attention Weight로 Value를 가중합해 Attention Output을 만듭니다.
    attention_output = tf.matmul(weights, v)

    # Residual Connection과 Layer Normalization을 적용합니다.
    out_1 = layer_norm_1(x + attention_output)

    # Feed Forward Network를 통과합니다.
    ffn_out = ffn_layer2(ffn_layer1(out_1))

    # 다시 Residual Connection과 Layer Normalization을 적용합니다.
    out_2 = layer_norm_2(out_1 + ffn_out)

    # 마지막 토큰의 벡터를 문장 전체 표현처럼 사용합니다.
    last_token = out_2[:, -1, :]

    # sigmoid 출력으로 긍정일 확률을 계산합니다.
    return classification_layer(last_token)


## 2.13 학습 가능한 변수 수집

Keras 레이어는 처음 데이터를 통과할 때 실제 가중치를 생성합니다. 그래서 학습 가능한 변수를 모으기 전에 `transformer_forward(input_ids)`를 한 번 호출합니다.

원본 노트북에서는 `_ = transformer_forward(input_ids)`처럼 결과를 버립니다. 여기서 `_`는 `결과값을 사용하지 않겠다`는 관습적 표현입니다.


In [ ]:
# 레이어 내부 가중치를 생성하기 위해 입력을 한 번 통과시킵니다.
_ = transformer_forward(input_ids)

# 학습할 변수들을 모두 하나의 리스트로 모읍니다.
trainable_vars = (
    token_embedding_layer.trainable_variables +
    pos_embedding_layer.trainable_variables +
    W_q.trainable_variables +
    W_k.trainable_variables +
    W_v.trainable_variables +
    layer_norm_1.trainable_variables +
    ffn_layer1.trainable_variables +
    ffn_layer2.trainable_variables +
    layer_norm_2.trainable_variables +
    classification_layer.trainable_variables
)

# 학습 가능한 변수 개수를 확인합니다.
print("학습 가능한 변수 개수:", len(trainable_vars))


## 2.14 학습 루프

이제 50번의 epoch 동안 모델을 학습합니다.

학습 흐름:

1. `GradientTape`로 자동미분 구간을 시작합니다.
2. 현재 모델로 예측값을 계산합니다.
3. 실제 라벨과 예측값 사이의 binary crossentropy 손실을 계산합니다.
4. 손실을 줄이기 위한 gradient를 계산합니다.
5. Adam optimizer로 모델 가중치를 업데이트합니다.
6. 10 epoch마다 손실값을 출력합니다.

이 예제는 데이터가 매우 작기 때문에 일반화 성능보다 구조 이해가 목적입니다.


In [ ]:
# Adam optimizer를 생성합니다.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.02)

# 50번 반복 학습합니다.
for epoch in range(1, 51):
    # 자동미분을 위해 GradientTape를 사용합니다.
    with tf.GradientTape() as tape:
        # 현재 모델의 예측값을 계산합니다.
        predictions = transformer_forward(input_ids)

        # 이진 분류 손실값을 계산합니다.
        loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(labels, predictions))

    # 손실값에 대한 각 변수의 gradient를 계산합니다.
    gradients = tape.gradient(loss, trainable_vars)

    # gradient를 변수에 적용해 가중치를 업데이트합니다.
    optimizer.apply_gradients(zip(gradients, trainable_vars))

    # 1번째 epoch와 10의 배수 epoch마다 손실을 출력합니다.
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/50 -> Loss: {loss.numpy():.4f}")

print("학습 완료")


## 2.15 새 문장 추론

학습 후에는 모델이 보지 못한 새 조합의 문장으로 테스트합니다.

원본 노트북의 테스트 문장:

| 문장 | 기대 |
|---|---|
| `i liked this movie` | liked가 긍정 단어이므로 긍정 기대 |
| `i hate this film` | hate가 부정 단어이므로 부정 기대 |

주의할 점은 이 예제의 단어 사전은 학습 문장에 등장한 단어만 처리합니다. 사전에 없는 단어가 들어오면 별도의 `<UNK>` 처리가 필요하지만, 원본 예제에서는 테스트 문장도 기존 단어만 사용합니다.


In [ ]:
# 학습에 직접 등장하지 않은 새 조합의 문장입니다.
test_sentences = [
    "i liked this movie",
    "i hate this film"
]

# 테스트 문장을 토큰 ID 배열로 변환합니다.
test_input_data = []
for sentence in test_sentences:
    ids = [vocab[word] for word in sentence.split()]
    test_input_data.append(ids)

# TensorFlow 텐서로 변환합니다.
test_input_ids = tf.constant(test_input_data, dtype=tf.int32)

# 학습된 Transformer 블록으로 예측합니다.
final_predictions = transformer_forward(test_input_ids)

# 예측 확률을 긍정/부정 라벨로 바꿔 출력합니다.
for i, sentence in enumerate(test_sentences):
    prob = final_predictions.numpy()[i][0]
    result = "긍정 (Positive)" if prob > 0.5 else "부정 (Negative)"
    print(f"입력 문장: {sentence}")
    print(f"예측 확률: {prob:.4f} -> 최종 결과: {result}")


## 2.16 Keras MultiHeadAttention을 사용하는 클래스형 구현

`LLM/llm.ipynb` 앞부분에는 `TransformerClassifier` 클래스로 모델을 정의하는 예제도 들어 있습니다. 이 방식은 직접 Q/K/V를 계산하는 대신 Keras의 `layers.MultiHeadAttention`을 사용합니다.

주요 구성은 다음과 같습니다.

- `token_emb_layer`: 토큰 임베딩
- `pos_emb_layer`: 위치 임베딩
- `MultiHeadAttention(num_heads=2, key_dim=d_model)`: 멀티헤드 어텐션
- `LayerNormalization`: Add & Norm
- `Dense`: Feed Forward Network
- `Dense(units=1, activation='sigmoid')`: 이진 분류 출력

실제 프로젝트에서는 함수형 직접 구현보다 이런 클래스형 모델이 재사용과 저장에 더 편리합니다.


In [ ]:
# Keras MultiHeadAttention을 사용하는 Transformer 분류기 클래스입니다.
class TransformerClassifier(tf.keras.Model):
    # 모델 생성 시 필요한 설정값을 받습니다.
    def __init__(self, vocab_size, seq_len, d_model):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.d_model = d_model

        # 토큰 임베딩과 위치 임베딩을 정의합니다.
        self.token_emb_layer = layers.Embedding(input_dim=vocab_size, output_dim=d_model, name="token_emb")
        self.pos_emb_layer = layers.Embedding(input_dim=seq_len, output_dim=d_model, name="pos_emb")
        self.position_indices = tf.range(start=0, limit=seq_len, delta=1)

        # Keras 내장 Multi-Head Attention을 사용합니다.
        self.transformer_attention = layers.MultiHeadAttention(num_heads=2, key_dim=d_model, name="attention")

        # Add & Norm을 위한 정규화 층입니다.
        self.layer_norm_1 = layers.LayerNormalization(epsilon=1e-6, name="ln_1")
        self.layer_norm_2 = layers.LayerNormalization(epsilon=1e-6, name="ln_2")

        # Feed Forward Network입니다.
        self.ffn_layer1 = layers.Dense(units=32, activation='relu', name="ffn_1")
        self.ffn_layer2 = layers.Dense(units=d_model, name="ffn_2")

        # 최종 분류 헤드입니다.
        self.classification_layer = layers.Dense(units=1, activation='sigmoid', name="clf_head")

    # 모델이 호출될 때 실행되는 정방향 계산입니다.
    def call(self, ids):
        w_emb = self.token_emb_layer(ids)
        p_emb = self.pos_emb_layer(self.position_indices)
        x = w_emb + tf.expand_dims(p_emb, axis=0)

        attention_output = self.transformer_attention(query=x, key=x, value=x)
        out_1 = self.layer_norm_1(x + attention_output)

        ffn_out = self.ffn_layer2(self.ffn_layer1(out_1))
        out_2 = self.layer_norm_2(out_1 + ffn_out)

        last_token = out_2[:, -1, :]
        return self.classification_layer(last_token)


## 2.17 직접 구현과 내장 레이어 구현의 차이

| 방식 | 장점 | 단점 |
|---|---|---|
| 직접 Q/K/V 구현 | Attention 계산 원리를 자세히 볼 수 있음 | 코드가 길고 실수하기 쉬움 |
| `MultiHeadAttention` 사용 | 실제 프로젝트에 가깝고 간결함 | 내부 계산이 감춰져 처음 학습할 때 덜 직관적일 수 있음 |

학습 순서는 직접 구현으로 원리를 이해한 뒤, 내장 레이어로 실전형 모델을 만드는 방식이 좋습니다.


## 2.18 Transformer 기반 모델 개발 구조

PDF 2.3은 Transformer 기반 모델 개발 구조를 보여줍니다. 실무 흐름으로 정리하면 다음과 같습니다.

1. 문제 정의: 분류, 생성, 요약, 번역, 질의응답 등
2. 데이터 준비: 원문 텍스트와 라벨 또는 문서 자료 준비
3. 토크나이저 선택: 모델에 맞는 토크나이저 사용
4. 모델 선택: BERT, GPT, T5, Gemma, Llama 등
5. 사전학습 모델 로드: Hugging Face 등에서 불러오기
6. 필요 시 Fine-tuning
7. 추론 Pipeline 구성
8. 저장과 배포: Hugging Face Hub, GitHub, API 서버 등
9. LangChain/RAG/Gradio 같은 애플리케이션으로 확장

`llm.ipynb`는 Tiny Transformer를 직접 만들고, 이후 가중치를 `safetensors`로 저장해 Hugging Face에 올리는 실습까지 포함합니다. 이 부분은 3장에서 자세히 다룹니다.


## 2.19 자주 발생하는 오류와 점검 포인트

### 1. shape mismatch 오류

Transformer에서는 텐서 shape이 중요합니다.

- 입력 shape: `(batch_size, seq_len)`
- 임베딩 후 shape: `(batch_size, seq_len, d_model)`
- Attention score shape: `(batch_size, seq_len, seq_len)`
- 분류 출력 shape: `(batch_size, 1)`

오류가 나면 각 단계의 shape을 `print(tensor.shape)`로 확인합니다.

### 2. 사전에 없는 단어 오류

테스트 문장에 학습 단어 사전에 없는 단어가 있으면 `vocab[word]`에서 KeyError가 발생합니다. 실제 모델에서는 `<UNK>` 토큰을 사용합니다.

### 3. 손실이 잘 줄지 않는 경우

가능한 원인:

- 데이터가 너무 적음
- learning rate가 너무 큼 또는 작음
- 모델 구조가 너무 단순함
- random seed에 따른 초기값 차이

이 예제는 구조 이해가 목적이므로 높은 정확도보다 각 연산 흐름을 이해하는 데 집중합니다.


## 2.20 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- Transformer는 문장 내 모든 단어 간 관계를 Attention으로 계산합니다.
- Self-Attention은 같은 문장 안의 단어들이 서로 얼마나 관련 있는지 계산합니다.
- Q는 궁금한 것, K는 비교 대상의 특징, V는 실제 전달할 정보입니다.
- Attention 수식은 `softmax(QK^T / sqrt(d_k))V`입니다.
- Encoder는 입력 문장을 이해하고 벡터 표현으로 만듭니다.
- Decoder는 벡터 표현을 바탕으로 문장을 생성합니다.
- 위치 인코딩은 단어 순서를 알려주는 역할을 합니다.
- Add & Norm은 잔차 연결과 정규화로 학습을 안정화합니다.
- Feed Forward Network는 각 토큰 벡터를 더 깊게 변환합니다.
- BERT는 양방향 문맥 이해에 강하고, GPT는 다음 토큰 생성에 강합니다.
- 작은 감정 분류 예제를 통해 Transformer의 입력, Attention, 학습, 추론 흐름을 직접 구현했습니다.

다음 장에서는 Hugging Face와 Transformer Pipeline을 사용해 사전학습 모델을 더 간단하게 불러오고 실행하는 방법을 다룹니다.
